# LAB 1: Implementation of Autoencoder for Image Generation on MNIST Dataset

**Mapped CO:** CO1

## Aim
To understand autoencoder architecture and use a convolutional autoencoder to reconstruct MNIST handwritten digit images.

## Theory
An autoencoder is a neural network trained to reconstruct its input. It contains:
- **Encoder:** extracts features and compresses the image.
- **Latent representation:** compact feature maps learned by the encoder.
- **Decoder:** reconstructs an image from the latent representation.

`MaxPooling2D` reduces spatial dimensions. `UpSampling2D` increases spatial dimensions. The model is trained with the original image as both input and target.

## Pseudocode
1. Load MNIST.
2. Normalize images and add the grayscale channel.
3. Build an encoder using convolution and max-pooling.
4. Build a decoder using convolution and up-sampling.
5. Combine them into an autoencoder.
6. Train using reconstruction loss.
7. Plot loss and compare original/reconstructed images.
8. Try decoding random latent vectors and save the model.

## Cell 1 — Install and import libraries

In [ ]:
%pip install -q tensorflow matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## Cell 2 — Load and inspect MNIST

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Testing images:", x_test.shape)

plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(str(i))
    plt.axis("off")
plt.suptitle("Sample MNIST images")
plt.tight_layout()
plt.show()

## Cell 3 — Preprocess the images

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)
print("Pixel range:", float(x_train.min()), "to", float(x_train.max()))

## Cell 4 — Define the encoder

In [ ]:
input_img = layers.Input(shape=(28, 28, 1), name="input_image")

x = layers.Conv2D(32, (3, 3), activation="relu", padding="same",
                  name="encoder_conv1")(input_img)
x = layers.MaxPooling2D((2, 2), padding="same",
                        name="encoder_pool1")(x)   # 14 × 14 × 32
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same",
                  name="encoder_conv2")(x)
x = layers.MaxPooling2D((2, 2), padding="same",
                        name="encoder_pool2")(x)   # 7 × 7 × 64
latent = layers.Conv2D(16, (3, 3), activation="relu", padding="same",
                       name="latent_representation")(x)  # 7 × 7 × 16

encoder = models.Model(input_img, latent, name="Encoder")
encoder.summary()

## Cell 5 — Define the decoder

In [ ]:
latent_input = layers.Input(shape=(7, 7, 16), name="latent_input")

x = layers.Conv2D(64, (3, 3), activation="relu", padding="same",
                  name="decoder_conv1")(latent_input)
x = layers.UpSampling2D((2, 2), name="decoder_upsample1")(x)  # 14 × 14
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same",
                  name="decoder_conv2")(x)
x = layers.UpSampling2D((2, 2), name="decoder_upsample2")(x)  # 28 × 28
output_img = layers.Conv2D(1, (3, 3), activation="sigmoid",
                           padding="same", name="reconstructed_image")(x)

decoder = models.Model(latent_input, output_img, name="Decoder")
decoder.summary()

## Cell 6 — Build and compile the autoencoder

In [ ]:
autoencoder = models.Model(
    inputs=input_img,
    outputs=decoder(encoder(input_img)),
    name="MNIST_Autoencoder"
)

autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()

## Cell 7 — Plot the model architecture

In [ ]:
try:
    tf.keras.utils.plot_model(
        autoencoder,
        to_file="autoencoder_architecture.png",
        show_shapes=True,
        show_layer_names=True
    )
    from IPython.display import Image, display
    display(Image(filename="autoencoder_architecture.png"))
except Exception as exc:
    print("Architecture image could not be generated:", exc)
    print("The model is still available; inspect autoencoder.summary().")

## Cell 8 — Train the autoencoder

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

history = autoencoder.fit(
    x_train,
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test, x_test),
    verbose=1
)

## Cell 9 — Plot training and validation loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Autoencoder Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 10 — Compare original and reconstructed images

In [ ]:
reconstructed = autoencoder.predict(x_test[:10], verbose=0)

plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Original")

    plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Reconstructed")

plt.suptitle("Original vs Reconstructed MNIST Images")
plt.tight_layout()
plt.show()

## Cell 11 — Evaluate reconstruction loss

In [ ]:
test_loss = autoencoder.evaluate(x_test, x_test, verbose=0)
mse = np.mean((x_test[:10] - reconstructed) ** 2)

print("Test reconstruction loss:", round(float(test_loss), 5))
print("MSE for first 10 test images:", round(float(mse), 6))

## Cell 12 — Decode random latent vectors

A standard autoencoder is trained for reconstruction, not to make its latent vectors follow a standard normal distribution. Random outputs may therefore look noisy or unlike digits. This cell demonstrates that limitation.

In [ ]:
np.random.seed(42)
random_latent = np.random.normal(
    loc=0.0, scale=1.0, size=(10, 7, 7, 16)
).astype("float32")

random_images = decoder.predict(random_latent, verbose=0)

plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(random_images[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle("Decoder outputs from random latent tensors")
plt.tight_layout()
plt.show()

## Cell 13 — Decode latent representations from real images

In [ ]:
encoded = encoder.predict(x_test[:1000], verbose=0)
indices = np.linspace(0, len(encoded) - 1, 10, dtype=int)
decoded_learned = decoder.predict(encoded[indices], verbose=0)

plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(decoded_learned[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle("Reconstructions from learned latent representations")
plt.tight_layout()
plt.show()

## Cell 14 — Save the models

In [ ]:
autoencoder.save("mnist_autoencoder.keras")
encoder.save("mnist_encoder.keras")
decoder.save("mnist_decoder.keras")

print("Saved autoencoder, encoder, and decoder.")

In [ ]:
from IPython.display import FileLink, display

display(FileLink("mnist_autoencoder.keras"))
display(FileLink("mnist_encoder.keras"))
display(FileLink("mnist_decoder.keras"))

## Architecture summary
**Input:** 28×28×1  
**Encoder:** Conv2D(32) → MaxPooling → Conv2D(64) → MaxPooling → Conv2D(16)  
**Latent feature map:** 7×7×16  
**Decoder:** Conv2D(64) → UpSampling → Conv2D(32) → UpSampling → Conv2D(1, sigmoid)  
**Output:** 28×28×1

## Result and conclusion
A convolutional autoencoder was implemented on MNIST. The encoder learns compressed feature maps and the decoder reconstructs the original images. Training and validation loss are plotted and reconstruction results are inspected. Random latent sampling is not guaranteed to produce meaningful digits for a conventional autoencoder.

## Viva questions
**1. What is an autoencoder?** A neural network that learns to reconstruct its input through a latent representation.

**2. What are the encoder and decoder?** The encoder compresses input features; the decoder reconstructs the image from them.

**3. Why use a latent representation?** It stores a compact representation of important input features.

**4. What is MaxPooling?** A down-sampling operation that retains maximum values in local regions.

**5. What is UpSampling?** An operation that increases the spatial dimensions of feature maps.

**6. Why is the target image the same as the input?** The model is trained to reconstruct the input rather than predict a label.

**7. Which loss function is used?** Binary cross-entropy between normalized input and reconstructed pixels.

## Screenshots for the lab record
Capture the encoder and decoder summaries, architecture image (if available), training/validation loss plot, and original-versus-reconstructed images after running the notebook.